In [1]:

# ======================================================================
# FINAL PAPER-READY KAGGLE PIPELINE
# Data-Adaptive Leakage-Aware OOF Stacking for DNS Attack Detection
#
# Experiments
#   1) CICIDS2017 DNS-associated traffic -> Binary: Benign vs Attack
#   2) Simulated DNS dataset            -> Multiclass:
#                                         Benign / DNS Spoofing /
#                                         Cache Poisoning
#
# Final workflow
#   Raw data
#   -> 80/20 stratified hold-out split
#   -> 5-fold CV on the 80% training set
#   -> fold-wise imputation/encoding
#   -> RF-based Top-20 feature selection
#   -> StandardScaler
#   -> SMOTE on training fold only
#   -> 10 candidate models
#   -> rank by CV Macro-F1 (tie: MCC, Balanced Accuracy, PR-AUC)
#   -> select best 3 models
#   -> OOF probabilities of selected 3
#   -> Logistic Regression meta-learner
#   -> final fit of selected 3 on complete 80% training set
#   -> untouched 20% test evaluation
#   -> SHAP explanation
# ======================================================================

# If Kaggle reports a missing package, uncomment once:
# !pip install -q imbalanced-learn xgboost lightgbm catboost shap

import gc
import json
import random
import time
import warnings
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc
)

from imblearn.over_sampling import SMOTE, RandomOverSampler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap

warnings.filterwarnings("ignore")

# ======================================================================
# 1. CONFIGURATION
# ======================================================================

SEED = 42
TEST_SIZE = 0.20
N_SPLITS = 5
TOP_K = 20

# CICIDS2017 has extremely few attack observations after DNS/port-53
# filtering. Fully balancing it with SMOTE can create an enormous number
# of synthetic rows. A capped minority/majority ratio is therefore used
# for the binary benchmark. Change to 1.0 only if you explicitly want
# full 1:1 SMOTE and have sufficient compute.
BINARY_SMOTE_RATIO = 0.25
MULTICLASS_SMOTE_STRATEGY = "auto"

# SHAP is intentionally calculated on a sample for practical runtime.
SHAP_SAMPLE_SIZE = 1500

# Actual uploaded filenames
PRIMARY_FILENAME = "CICIDS2017 Dataset.csv"
SECONDARY_FILENAME = "CDAD_multiclass.csv"

PRIMARY_LABEL = "Label"
SECONDARY_LABEL = "Multiclass_Label"

# CICIDS DNS association
SOURCE_PORT_COL = "Source Port"
DEST_PORT_COL = "Destination Port"
DNS_PORT = 53

SECONDARY_CLASS_NAMES = {
    0: "Benign",
    1: "DNS Spoofing",
    2: "Cache Poisoning"
}

OUTPUT_ROOT = Path("/kaggle/working/paper_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)


# ======================================================================
# 2. KAGGLE FILE HELPERS
# ======================================================================

def find_input_file(filename):
    matches = list(Path("/kaggle/input").rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"'{filename}' was not found under /kaggle/input.\n"
            "Check the filename after adding the dataset to Kaggle."
        )
    if len(matches) > 1:
        print(f"Multiple matches found; using {matches[0]}")
    return matches[0]


def list_csv_inputs():
    print("CSV files available in Kaggle input:")
    for p in Path("/kaggle/input").rglob("*.csv"):
        print(" -", p)


# ======================================================================
# 3. PAPER-READY OUTPUT HELPERS
# ======================================================================

def create_output_dirs(key):
    root = OUTPUT_ROOT / key
    dirs = {
        "root": root,
        "figures": root / "figures",
        "tables": root / "tables",
        "reports": root / "reports"
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def save_figure(fig, path_without_extension):
    """Save publication figures as 600-DPI PNG and vector PDF."""
    fig.savefig(
        str(path_without_extension) + ".png",
        dpi=600,
        bbox_inches="tight"
    )
    fig.savefig(
        str(path_without_extension) + ".pdf",
        bbox_inches="tight"
    )
    plt.close(fig)


def save_table(df, path_without_extension, index=False):
    """Save every paper table as CSV and ready-to-copy LaTeX."""
    df.to_csv(str(path_without_extension) + ".csv", index=index)

    latex = df.to_latex(
        index=index,
        escape=True,
        float_format=lambda x: f"{x:.6f}"
        if isinstance(x, (float, np.floating)) else str(x)
    )
    Path(str(path_without_extension) + ".tex").write_text(
        latex, encoding="utf-8"
    )


# ======================================================================
# 4. DATASET-SPECIFIC CLEANING
# ======================================================================

def normalize_colname(name):
    return (
        str(name).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )


def remove_non_behavioral_and_target_proxy_fields(df, label_col):
    """
    Automatic leakage guard, not manual feature selection.

    It removes:
      - direct target proxies (e.g., Attack_Type)
      - experiment/session/scenario IDs
      - raw timestamps
      - raw domain/IP identity strings

    The actual predictive Top-20 features are still selected automatically
    by Random Forest within the training data only.
    """
    direct_proxies = {
        "attack_type",
        "class",
        "target",
        "target_label",
        "binary_label"
    }

    identity_fields = {
        "experiment_id",
        "session_id",
        "scenario_id",
        "record_timestamp",
        "timestamp",
        "flow_id",
        "domain_queried",
        "source_ip",
        "destination_ip",
        "src_ip",
        "dst_ip"
    }

    drop_cols = []
    for col in df.columns:
        if col == label_col:
            continue
        n = normalize_colname(col)
        if n in direct_proxies or n in identity_fields:
            drop_cols.append(col)

    if drop_cols:
        print("Automatically excluded non-behavioral / leakage-prone fields:")
        for c in drop_cols:
            print("  -", c)

    return df.drop(columns=drop_cols, errors="ignore"), drop_cols


def basic_clean(df, label_col):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    if label_col not in df.columns:
        raise ValueError(
            f"Target column '{label_col}' not found.\n"
            f"Columns: {list(df.columns)}"
        )

    original_n = len(df)

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df[df[label_col].notna()].copy()

    # Exact duplicates are removed before train/test splitting so the same
    # record cannot occur in both sets.
    df = df.drop_duplicates().reset_index(drop=True)

    df, excluded = remove_non_behavioral_and_target_proxy_fields(
        df, label_col
    )

    # Convert numeric-looking text columns to numeric.
    for col in df.columns:
        if col == label_col:
            continue

        if df[col].dtype == "object":
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().mean() >= 0.95:
                df[col] = converted

    cleaning_info = {
        "rows_before_cleaning": int(original_n),
        "rows_after_cleaning": int(len(df)),
        "duplicates_or_invalid_label_rows_removed": int(original_n - len(df)),
        "automatic_excluded_fields": excluded
    }

    return df, cleaning_info


def load_primary_dataset(path):
    """
    CICIDS2017:
    DNS-associated = Source Port 53 OR Destination Port 53.
    Original BENIGN -> 0; all original attack labels -> 1.
    """
    raw = pd.read_csv(path, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]

    mask = (
        pd.to_numeric(raw[SOURCE_PORT_COL], errors="coerce").eq(DNS_PORT)
        |
        pd.to_numeric(raw[DEST_PORT_COL], errors="coerce").eq(DNS_PORT)
    )
    dns = raw.loc[mask].copy().reset_index(drop=True)

    # Save original label distribution before binary remapping.
    original_label_distribution = (
        dns[PRIMARY_LABEL].astype(str).str.strip().value_counts().to_dict()
    )

    dns, clean_info = basic_clean(dns, PRIMARY_LABEL)

    y_text = dns[PRIMARY_LABEL].astype(str).str.strip()
    y = np.where(
        y_text.str.upper().eq("BENIGN"), 0, 1
    ).astype(int)

    X = dns.drop(columns=[PRIMARY_LABEL])
    class_names = ["Benign", "Attack"]

    clean_info["dns_filter"] = (
        f"({SOURCE_PORT_COL} == {DNS_PORT}) OR "
        f"({DEST_PORT_COL} == {DNS_PORT})"
    )
    clean_info["original_dns_label_distribution"] = (
        original_label_distribution
    )

    return X, y, class_names, clean_info


def load_secondary_dataset(path):
    raw = pd.read_csv(path, low_memory=False)
    raw.columns = [str(c).strip() for c in raw.columns]

    raw, clean_info = basic_clean(raw, SECONDARY_LABEL)

    y = pd.to_numeric(
        raw[SECONDARY_LABEL], errors="raise"
    ).astype(int).to_numpy()

    unexpected = sorted(
        set(np.unique(y)) - set(SECONDARY_CLASS_NAMES.keys())
    )
    if unexpected:
        raise ValueError(
            f"Unexpected secondary labels: {unexpected}. "
            "Update SECONDARY_CLASS_NAMES."
        )

    X = raw.drop(columns=[SECONDARY_LABEL])
    class_names = [
        SECONDARY_CLASS_NAMES[i]
        for i in sorted(SECONDARY_CLASS_NAMES)
    ]

    return X, y, class_names, clean_info


# ======================================================================
# 5. FOLD-WISE IMPUTATION + ENCODING
# ======================================================================

def build_encoder_imputer(X):
    """
    No scaling here.
    RF feature selection is intentionally performed BEFORE StandardScaler.
    """
    numeric_cols = X.select_dtypes(
        include=[np.number, "bool"]
    ).columns.tolist()

    categorical_cols = [
        c for c in X.columns if c not in numeric_cols
    ]

    transformers = []

    if numeric_cols:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])
        transformers.append(
            ("num", numeric_pipe, numeric_cols)
        )

    if categorical_cols:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ])
        transformers.append(
            ("cat", categorical_pipe, categorical_cols)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0
    )


def clean_feature_names(names):
    return np.asarray(
        [str(x).split("__", 1)[-1] for x in names],
        dtype=object
    )


# ======================================================================
# 6. RF-BASED TOP-20 FEATURE SELECTION
# ======================================================================

def select_top20_rf(X_train, y_train, feature_names):
    selector = RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1,
        class_weight="balanced_subsample",
        max_features="sqrt"
    )
    selector.fit(X_train, y_train)

    importance = selector.feature_importances_
    k = min(TOP_K, len(feature_names))
    idx = np.argsort(importance)[::-1][:k]

    return {
        "indices": idx,
        "names": np.asarray(feature_names)[idx],
        "importance": importance[idx],
        "model": selector
    }


# ======================================================================
# 7. SCALING
# ======================================================================

def fit_and_apply_scaler(X_train_selected, X_other_selected):
    """
    StandardScaler is fitted ONLY on the current training data.
    Validation/test is transformed using the same fitted scaler.
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_selected)
    X_other_scaled = scaler.transform(X_other_selected)

    return X_train_scaled, X_other_scaled, scaler


# ======================================================================
# 8. TRAINING-ONLY SMOTE
# ======================================================================

def apply_training_smote(X_train, y_train, n_classes):
    """
    Binary CICIDS uses capped SMOTE because its DNS subset is extremely
    imbalanced. Multiclass uses standard automatic balancing.
    """
    counts = Counter(y_train)
    min_count = min(counts.values())

    if min_count <= 1:
        sampler = RandomOverSampler(random_state=SEED)

    elif n_classes == 2:
        # If minority is already above the configured ratio, no forced
        # oversampling beyond its current ratio is necessary.
        majority = max(counts.values())
        minority = min(counts.values())
        current_ratio = minority / majority

        if current_ratio >= BINARY_SMOTE_RATIO:
            return X_train, y_train, None

        sampler = SMOTE(
            random_state=SEED,
            k_neighbors=min(5, min_count - 1),
            sampling_strategy=BINARY_SMOTE_RATIO
        )

    else:
        sampler = SMOTE(
            random_state=SEED,
            k_neighbors=min(5, min_count - 1),
            sampling_strategy=MULTICLASS_SMOTE_STRATEGY
        )

    X_resampled, y_resampled = sampler.fit_resample(
        X_train, y_train
    )

    return X_resampled, y_resampled, sampler


# ======================================================================
# 9. TEN CANDIDATE BASE MODELS
# ======================================================================

def calibrated_linear_svm():
    base = LinearSVC(
        C=1.0,
        random_state=SEED,
        max_iter=10000
    )

    # Compatibility across sklearn versions.
    try:
        return CalibratedClassifierCV(
            estimator=base,
            cv=3,
            method="sigmoid"
        )
    except TypeError:
        return CalibratedClassifierCV(
            base_estimator=base,
            cv=3,
            method="sigmoid"
        )


def build_candidate_models(n_classes):
    binary = (n_classes == 2)

    xgb_params = dict(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        tree_method="hist",
        eval_metric="logloss" if binary else "mlogloss"
    )
    if not binary:
        xgb_params.update(
            objective="multi:softprob",
            num_class=n_classes
        )

    cat_params = dict(
        iterations=250,
        depth=7,
        learning_rate=0.05,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        loss_function="Logloss" if binary else "MultiClass"
    )

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=3000,
            random_state=SEED
        ),

        "SVM": calibrated_linear_svm(),

        "Random Forest": RandomForestClassifier(
            n_estimators=250,
            max_features="sqrt",
            random_state=SEED,
            n_jobs=-1
        ),

        "Extra Trees": ExtraTreesClassifier(
            n_estimators=250,
            max_features="sqrt",
            random_state=SEED,
            n_jobs=-1
        ),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=150,
            learning_rate=0.05,
            random_state=SEED
        ),

        "XGBoost": XGBClassifier(
            **xgb_params
        ),

        "LightGBM": LGBMClassifier(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=63,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1
        ),

        "CatBoost": CatBoostClassifier(
            **cat_params
        ),

        # Shallow feed-forward network
        "MLP": MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=256,
            learning_rate_init=1e-3,
            max_iter=220,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=15,
            random_state=SEED
        ),

        # Deeper feed-forward neural network
        "DNN": MLPClassifier(
            hidden_layer_sizes=(256, 128, 64, 32),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=256,
            learning_rate_init=5e-4,
            max_iter=280,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=18,
            random_state=SEED
        )
    }

    return models


# ======================================================================
# 10. METRICS
# ======================================================================

def calculate_metrics(
    y_true,
    y_pred,
    y_prob,
    n_classes
):
    result = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "Precision_Macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Recall_Macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "F1_Macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }

    try:
        if n_classes == 2:
            result["ROC_AUC"] = roc_auc_score(
                y_true, y_prob[:, 1]
            )
            result["PR_AUC"] = average_precision_score(
                y_true, y_prob[:, 1]
            )
        else:
            result["ROC_AUC"] = roc_auc_score(
                y_true,
                y_prob,
                average="macro",
                multi_class="ovr"
            )
            y_bin = label_binarize(
                y_true,
                classes=np.arange(n_classes)
            )
            result["PR_AUC"] = average_precision_score(
                y_bin,
                y_prob,
                average="macro"
            )
    except Exception:
        result["ROC_AUC"] = np.nan
        result["PR_AUC"] = np.nan

    return result


def probability_for_meta(prob, n_classes):
    # Binary: only P(Attack) is needed from each base learner.
    # Multiclass: retain the full class-probability vector.
    if n_classes == 2:
        return prob[:, [1]]
    return prob


# ======================================================================
# 11. PAPER-READY FIGURES
# ======================================================================

def plot_top20_features(feature_table, title, output_path):
    temp = feature_table.sort_values(
        "RF_Importance", ascending=True
    )

    fig, ax = plt.subplots(figsize=(8.5, 7.2))
    ax.barh(
        temp["Feature"],
        temp["RF_Importance"]
    )
    ax.set_xlabel("Random Forest Feature Importance")
    ax.set_title(title)
    fig.tight_layout()
    save_figure(fig, output_path)


def plot_smote_distribution(
    y_before,
    y_after,
    class_names,
    title,
    output_path
):
    before = np.bincount(
        y_before,
        minlength=len(class_names)
    )
    after = np.bincount(
        y_after,
        minlength=len(class_names)
    )

    x = np.arange(len(class_names))
    width = 0.38

    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    ax.bar(
        x - width / 2,
        before,
        width,
        label="Before SMOTE"
    )
    ax.bar(
        x + width / 2,
        after,
        width,
        label="After SMOTE"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        class_names,
        rotation=15,
        ha="right"
    )
    ax.set_ylabel("Number of training samples")
    ax.set_title(title)
    ax.legend()

    fig.tight_layout()
    save_figure(fig, output_path)


def plot_candidate_ranking(
    ranking_df,
    title,
    output_path
):
    plot_df = ranking_df.sort_values(
        "F1_Macro_mean",
        ascending=True
    )

    fig, ax = plt.subplots(figsize=(9.2, 6.2))
    ax.barh(
        plot_df["Model"],
        plot_df["F1_Macro_mean"]
    )

    for y_pos, (_, row) in enumerate(
        plot_df.iterrows()
    ):
        ax.text(
            row["F1_Macro_mean"],
            y_pos,
            f" {row['F1_Macro_mean']:.4f}",
            va="center"
        )

    ax.set_xlabel("Mean 5-Fold Macro-F1")
    ax.set_title(title)
    fig.tight_layout()
    save_figure(fig, output_path)


def plot_cv_top3(
    cv_fold_df,
    selected_models,
    title,
    output_path
):
    sub = cv_fold_df[
        cv_fold_df["Model"].isin(selected_models)
    ]

    pivot = sub.pivot(
        index="Fold",
        columns="Model",
        values="F1_Macro"
    )

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    pivot.plot(marker="o", ax=ax)
    ax.set_ylabel("Macro-F1")
    ax.set_xlabel("Fold")
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, output_path)


def plot_final_comparison(
    final_df,
    title,
    output_path
):
    metrics = [
        "Accuracy",
        "Balanced_Accuracy",
        "F1_Macro",
        "MCC"
    ]

    plot_df = final_df.set_index("Model")[metrics]

    fig, ax = plt.subplots(figsize=(9.2, 5.5))
    plot_df.plot(kind="bar", ax=ax)
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=15)
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, output_path)


def plot_confusion(
    y_true,
    y_pred,
    class_names,
    title,
    output_path,
    normalized=False
):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        normalize="true" if normalized else None
    )

    fig, ax = plt.subplots(figsize=(6.3, 5.4))
    image = ax.imshow(cm)
    fig.colorbar(image, ax=ax)

    ax.set_xticks(
        np.arange(len(class_names))
    )
    ax.set_yticks(
        np.arange(len(class_names))
    )
    ax.set_xticklabels(
        class_names,
        rotation=25,
        ha="right"
    )
    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(title)

    threshold = np.nanmax(cm) / 2

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text = (
                f"{cm[i, j]:.3f}"
                if normalized
                else f"{int(cm[i, j]):,}"
            )
            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                color=(
                    "white"
                    if cm[i, j] > threshold
                    else "black"
                )
            )

    fig.tight_layout()
    save_figure(fig, output_path)


def plot_roc(
    y_true,
    probabilities,
    class_names,
    title,
    output_path
):
    n_classes = len(class_names)

    fig, ax = plt.subplots(figsize=(7.2, 6.0))

    if n_classes == 2:
        for name, prob in probabilities.items():
            fpr, tpr, _ = roc_curve(
                y_true,
                prob[:, 1]
            )
            score = auc(fpr, tpr)
            ax.plot(
                fpr,
                tpr,
                label=f"{name} (AUC={score:.4f})"
            )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random"
        )

    else:
        y_bin = label_binarize(
            y_true,
            classes=np.arange(n_classes)
        )
        stack_prob = probabilities[
            "Proposed Top-3 Stack"
        ]

        for c, cname in enumerate(class_names):
            fpr, tpr, _ = roc_curve(
                y_bin[:, c],
                stack_prob[:, c]
            )
            score = auc(fpr, tpr)

            ax.plot(
                fpr,
                tpr,
                label=f"{cname} (AUC={score:.4f})"
            )

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(
        loc="lower right",
        fontsize=8
    )

    fig.tight_layout()
    save_figure(fig, output_path)


def plot_pr(
    y_true,
    probabilities,
    class_names,
    title,
    output_path
):
    n_classes = len(class_names)

    fig, ax = plt.subplots(figsize=(7.2, 6.0))

    if n_classes == 2:
        for name, prob in probabilities.items():
            precision, recall, _ = (
                precision_recall_curve(
                    y_true,
                    prob[:, 1]
                )
            )
            ap = average_precision_score(
                y_true,
                prob[:, 1]
            )

            ax.plot(
                recall,
                precision,
                label=f"{name} (AP={ap:.4f})"
            )

    else:
        y_bin = label_binarize(
            y_true,
            classes=np.arange(n_classes)
        )
        stack_prob = probabilities[
            "Proposed Top-3 Stack"
        ]

        for c, cname in enumerate(class_names):
            precision, recall, _ = (
                precision_recall_curve(
                    y_bin[:, c],
                    stack_prob[:, c]
                )
            )
            ap = average_precision_score(
                y_bin[:, c],
                stack_prob[:, c]
            )

            ax.plot(
                recall,
                precision,
                label=f"{cname} (AP={ap:.4f})"
            )

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend(
        loc="lower left",
        fontsize=8
    )

    fig.tight_layout()
    save_figure(fig, output_path)


def plot_meta_coefficients(
    meta_model,
    selected_models,
    n_classes,
    class_names,
    title,
    output_path
):
    coef = np.asarray(meta_model.coef_)

    if n_classes == 2:
        labels = np.asarray(selected_models)
        values = np.abs(coef[0])

    else:
        labels = []
        for model_name in selected_models:
            for cname in class_names:
                labels.append(
                    f"{model_name}: P({cname})"
                )
        labels = np.asarray(labels)
        values = np.mean(
            np.abs(coef),
            axis=0
        )

    order = np.argsort(values)

    fig, ax = plt.subplots(
        figsize=(9, max(5, len(labels) * 0.35))
    )
    ax.barh(
        labels[order],
        values[order]
    )
    ax.set_xlabel(
        "Absolute Logistic Regression coefficient"
    )
    ax.set_title(title)

    fig.tight_layout()
    save_figure(fig, output_path)


# ======================================================================
# 12. SHAP
# ======================================================================

TREE_MODEL_NAMES = {
    "Random Forest",
    "Extra Trees",
    "XGBoost",
    "LightGBM",
    "CatBoost"
}


def choose_shap_model(selected_models):
    """
    Prefer the highest-ranked selected tree model because TreeSHAP is fast
    and directly explains the selected traffic features.
    """
    for name in selected_models:
        if name in TREE_MODEL_NAMES:
            return name
    return selected_models[0]


def generate_shap_analysis(
    model,
    model_name,
    X_test,
    feature_names,
    class_names,
    dirs,
    dataset_key,
    dataset_title
):
    n = min(
        SHAP_SAMPLE_SIZE,
        len(X_test)
    )

    rng = np.random.default_rng(SEED)
    indices = rng.choice(
        len(X_test),
        size=n,
        replace=False
    )
    X_sample = np.asarray(X_test)[indices]

    try:
        if model_name in TREE_MODEL_NAMES:
            explainer = shap.TreeExplainer(model)
            raw_values = explainer.shap_values(
                X_sample
            )
        else:
            # Generic fallback for the unlikely case that none of the top
            # three models is tree-based.
            background_n = min(
                100,
                len(X_sample)
            )
            background = X_sample[:background_n]

            explainer = shap.Explainer(
                model.predict_proba,
                background,
                feature_names=list(feature_names)
            )

            explain_n = min(
                300,
                len(X_sample)
            )
            explanation = explainer(
                X_sample[:explain_n]
            )
            raw_values = explanation.values
            X_sample = X_sample[:explain_n]

        if isinstance(raw_values, list):
            values = np.stack(
                raw_values,
                axis=-1
            )
        elif hasattr(raw_values, "values"):
            values = np.asarray(
                raw_values.values
            )
        else:
            values = np.asarray(
                raw_values
            )

        if values.ndim == 2:
            mean_abs = np.mean(
                np.abs(values),
                axis=0
            )

        elif values.ndim == 3:
            mean_abs = np.mean(
                np.abs(values),
                axis=(0, 2)
            )

        else:
            raise ValueError(
                f"Unexpected SHAP shape: {values.shape}"
            )

        order = np.argsort(
            mean_abs
        )[::-1][:TOP_K]

        shap_table = pd.DataFrame({
            "Rank": np.arange(
                1,
                len(order) + 1
            ),
            "Feature": np.asarray(
                feature_names
            )[order],
            "Mean_Absolute_SHAP": mean_abs[
                order
            ]
        })

        save_table(
            shap_table,
            dirs["tables"]
            / f"{dataset_key}_table_08_shap_importance"
        )

        temp = shap_table.sort_values(
            "Mean_Absolute_SHAP",
            ascending=True
        )

        fig, ax = plt.subplots(
            figsize=(8.5, 7.2)
        )
        ax.barh(
            temp["Feature"],
            temp["Mean_Absolute_SHAP"]
        )
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title(
            f"{dataset_title}: SHAP Global Importance "
            f"({model_name})"
        )

        fig.tight_layout()
        save_figure(
            fig,
            dirs["figures"]
            / f"{dataset_key}_fig_10_shap_global_importance"
        )

        # Binary beeswarm when a simple 2D SHAP matrix is returned.
        if values.ndim == 2:
            plt.figure(figsize=(8.5, 7.0))
            shap.summary_plot(
                values,
                X_sample,
                feature_names=list(feature_names),
                max_display=TOP_K,
                show=False
            )
            plt.title(
                f"{dataset_title}: SHAP Summary ({model_name})"
            )
            plt.tight_layout()
            plt.savefig(
                dirs["figures"]
                / f"{dataset_key}_fig_11_shap_summary.png",
                dpi=600,
                bbox_inches="tight"
            )
            plt.savefig(
                dirs["figures"]
                / f"{dataset_key}_fig_11_shap_summary.pdf",
                bbox_inches="tight"
            )
            plt.close()

    except Exception as exc:
        print(
            f"SHAP warning for {dataset_title}: {exc}"
        )


# ======================================================================
# 13. CORE EXPERIMENT
# ======================================================================

def run_experiment(
    X,
    y,
    class_names,
    dataset_key,
    dataset_title,
    cleaning_info
):
    dirs = create_output_dirs(
        dataset_key
    )

    n_classes = len(class_names)

    print("\n" + "=" * 90)
    print(dataset_title)
    print("=" * 90)
    print(f"Records: {len(X):,}")
    print(f"Raw predictive columns: {X.shape[1]}")
    print("Classes:", class_names)
    print("Distribution:", dict(Counter(y)))

    # ------------------------------------------------------------------
    # A. 80/20 STRATIFIED HOLD-OUT SPLIT
    # ------------------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=SEED
    )

    # Basic dataset summary.
    summary_df = pd.DataFrame({
        "Item": [
            "Total records",
            "Training records (80%)",
            "Untouched test records (20%)",
            "Predictive columns before RF selection",
            "Selected features"
        ],
        "Value": [
            len(X),
            len(X_train),
            len(X_test),
            X.shape[1],
            TOP_K
        ]
    })

    save_table(
        summary_df,
        dirs["tables"]
        / f"{dataset_key}_table_01_dataset_summary"
    )

    class_rows = []
    for c, cname in enumerate(class_names):
        class_rows.append({
            "Class": cname,
            "Full_Dataset": int(
                np.sum(y == c)
            ),
            "Training_80pct": int(
                np.sum(y_train == c)
            ),
            "Test_20pct": int(
                np.sum(y_test == c)
            )
        })

    save_table(
        pd.DataFrame(class_rows),
        dirs["tables"]
        / f"{dataset_key}_table_02_class_distribution"
    )

    # ------------------------------------------------------------------
    # B. FIVE-FOLD CV FOR ALL 10 CANDIDATE MODELS
    # ------------------------------------------------------------------
    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    model_names = list(
        build_candidate_models(
            n_classes
        ).keys()
    )

    # Store OOF probabilities for ALL candidate models.
    oof_probabilities = {
        name: np.zeros(
            (len(X_train), n_classes),
            dtype=float
        )
        for name in model_names
    }

    cv_rows = []
    selected_features_by_fold = []

    for fold, (tr_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):
        print(f"\nFold {fold}/{N_SPLITS}")

        X_tr = X_train.iloc[
            tr_idx
        ].copy()
        X_val = X_train.iloc[
            val_idx
        ].copy()

        y_tr = y_train[
            tr_idx
        ]
        y_val = y_train[
            val_idx
        ]

        # 1) Fit imputer/encoder ONLY on training fold.
        encoder = build_encoder_imputer(
            X_tr
        )

        X_tr_encoded = encoder.fit_transform(
            X_tr
        )
        X_val_encoded = encoder.transform(
            X_val
        )

        feature_names = clean_feature_names(
            encoder.get_feature_names_out()
        )

        # 2) RF Top-20 ONLY on training fold.
        fs = select_top20_rf(
            X_tr_encoded,
            y_tr,
            feature_names
        )

        selected_features_by_fold.append(
            list(fs["names"])
        )

        X_tr_20 = X_tr_encoded[
            :,
            fs["indices"]
        ]
        X_val_20 = X_val_encoded[
            :,
            fs["indices"]
        ]

        # 3) StandardScaler fit ONLY on selected training-fold features.
        X_tr_scaled, X_val_scaled, scaler = (
            fit_and_apply_scaler(
                X_tr_20,
                X_val_20
            )
        )

        # 4) SMOTE ONLY on scaled training fold.
        X_bal, y_bal, sampler = (
            apply_training_smote(
                X_tr_scaled,
                y_tr,
                n_classes
            )
        )

        print(
            "  Before SMOTE:",
            dict(Counter(y_tr))
        )
        print(
            "  After SMOTE :",
            dict(Counter(y_bal))
        )

        # 5) Evaluate all 10 candidate models.
        candidate_models = (
            build_candidate_models(
                n_classes
            )
        )

        for model_name, model in candidate_models.items():
            start = time.perf_counter()

            model.fit(
                X_bal,
                y_bal
            )

            fit_seconds = (
                time.perf_counter()
                - start
            )

            start = time.perf_counter()

            prob = model.predict_proba(
                X_val_scaled
            )

            inference_seconds = (
                time.perf_counter()
                - start
            )

            pred = np.argmax(
                prob,
                axis=1
            )

            # OOF predictions from samples unseen by this fold's model.
            oof_probabilities[
                model_name
            ][val_idx] = prob

            metrics = calculate_metrics(
                y_val,
                pred,
                prob,
                n_classes
            )

            cv_rows.append({
                "Fold": fold,
                "Model": model_name,
                **metrics,
                "Training_Time_sec": fit_seconds,
                "Validation_Inference_Time_sec": inference_seconds
            })

            print(
                f"  {model_name:<20} "
                f"F1={metrics['F1_Macro']:.5f}  "
                f"MCC={metrics['MCC']:.5f}  "
                f"Acc={metrics['Accuracy']:.5f}"
            )

            del model
            gc.collect()

        del (
            X_tr_encoded,
            X_val_encoded,
            X_tr_20,
            X_val_20,
            X_tr_scaled,
            X_val_scaled,
            X_bal,
            y_bal,
            candidate_models
        )
        gc.collect()

    cv_fold_df = pd.DataFrame(
        cv_rows
    )

    save_table(
        cv_fold_df,
        dirs["tables"]
        / f"{dataset_key}_table_03_candidate_fold_metrics"
    )

    # ------------------------------------------------------------------
    # C. CV SUMMARY + AUTOMATIC TOP-3 SELECTION
    # ------------------------------------------------------------------
    cv_summary = (
        cv_fold_df
        .groupby("Model")
        .agg(
            Accuracy_mean=("Accuracy", "mean"),
            Accuracy_std=("Accuracy", "std"),
            Balanced_Accuracy_mean=("Balanced_Accuracy", "mean"),
            Balanced_Accuracy_std=("Balanced_Accuracy", "std"),
            Precision_Macro_mean=("Precision_Macro", "mean"),
            Precision_Macro_std=("Precision_Macro", "std"),
            Recall_Macro_mean=("Recall_Macro", "mean"),
            Recall_Macro_std=("Recall_Macro", "std"),
            F1_Macro_mean=("F1_Macro", "mean"),
            F1_Macro_std=("F1_Macro", "std"),
            MCC_mean=("MCC", "mean"),
            MCC_std=("MCC", "std"),
            ROC_AUC_mean=("ROC_AUC", "mean"),
            ROC_AUC_std=("ROC_AUC", "std"),
            PR_AUC_mean=("PR_AUC", "mean"),
            PR_AUC_std=("PR_AUC", "std"),
            Training_Time_mean_sec=("Training_Time_sec", "mean"),
            Inference_Time_mean_sec=(
                "Validation_Inference_Time_sec",
                "mean"
            )
        )
        .reset_index()
    )

    # Primary criterion: Macro-F1.
    # Tie-breakers: MCC, Balanced Accuracy, PR-AUC.
    ranking_df = cv_summary.sort_values(
        by=[
            "F1_Macro_mean",
            "MCC_mean",
            "Balanced_Accuracy_mean",
            "PR_AUC_mean"
        ],
        ascending=False
    ).reset_index(drop=True)

    ranking_df.insert(
        0,
        "Rank",
        np.arange(
            1,
            len(ranking_df) + 1
        )
    )

    ranking_df["Selected_for_Stack"] = np.where(
        ranking_df["Rank"] <= 3,
        "Yes",
        "No"
    )

    selected_models = ranking_df.loc[
        ranking_df["Rank"] <= 3,
        "Model"
    ].tolist()

    save_table(
        ranking_df,
        dirs["tables"]
        / f"{dataset_key}_table_04_candidate_model_ranking"
    )

    selected_top3_table = ranking_df[
        ranking_df["Rank"] <= 3
    ].copy()

    save_table(
        selected_top3_table,
        dirs["tables"]
        / f"{dataset_key}_table_05_selected_top3"
    )

    print("\nSelected Top-3 models:")
    for i, model_name in enumerate(
        selected_models,
        start=1
    ):
        print(f"  {i}. {model_name}")

    plot_candidate_ranking(
        ranking_df,
        f"{dataset_title}: Candidate Model Ranking",
        dirs["figures"]
        / f"{dataset_key}_fig_01_candidate_ranking"
    )

    plot_cv_top3(
        cv_fold_df,
        selected_models,
        f"{dataset_title}: Selected Top-3 Across Five Folds",
        dirs["figures"]
        / f"{dataset_key}_fig_02_top3_cv_stability"
    )

    # ------------------------------------------------------------------
    # D. OOF META MATRIX FROM SELECTED TOP-3 ONLY
    # ------------------------------------------------------------------
    meta_parts = [
        probability_for_meta(
            oof_probabilities[name],
            n_classes
        )
        for name in selected_models
    ]

    Z_oof = np.hstack(
        meta_parts
    )

    meta_model = LogisticRegression(
        max_iter=3000,
        random_state=SEED
    )

    meta_model.fit(
        Z_oof,
        y_train
    )

    # Secondary CV ONLY in the OOF meta-space to give a descriptive
    # training-stage estimate for the proposed stack.
    meta_cv_prob = np.zeros(
        (len(y_train), n_classes),
        dtype=float
    )

    for tr_idx, val_idx in skf.split(
        Z_oof,
        y_train
    ):
        lr = clone(
            meta_model
        )

        lr.fit(
            Z_oof[tr_idx],
            y_train[tr_idx]
        )

        meta_cv_prob[val_idx] = (
            lr.predict_proba(
                Z_oof[val_idx]
            )
        )

    meta_cv_pred = np.argmax(
        meta_cv_prob,
        axis=1
    )

    meta_cv_metrics = calculate_metrics(
        y_train,
        meta_cv_pred,
        meta_cv_prob,
        n_classes
    )

    # ------------------------------------------------------------------
    # E. FINAL FIT ON COMPLETE 80% TRAINING PARTITION
    # ------------------------------------------------------------------
    final_encoder = build_encoder_imputer(
        X_train
    )

    X_train_encoded = final_encoder.fit_transform(
        X_train
    )
    X_test_encoded = final_encoder.transform(
        X_test
    )

    final_feature_names = clean_feature_names(
        final_encoder.get_feature_names_out()
    )

    # Final RF Top-20 fitted ONLY on full 80% training data.
    final_fs = select_top20_rf(
        X_train_encoded,
        y_train,
        final_feature_names
    )

    X_train_20 = X_train_encoded[
        :,
        final_fs["indices"]
    ]
    X_test_20 = X_test_encoded[
        :,
        final_fs["indices"]
    ]

    # Final scaler fitted ONLY on full 80% training data.
    (
        X_train_scaled,
        X_test_scaled,
        final_scaler
    ) = fit_and_apply_scaler(
        X_train_20,
        X_test_20
    )

    # Final training-only SMOTE.
    (
        X_train_bal,
        y_train_bal,
        final_sampler
    ) = apply_training_smote(
        X_train_scaled,
        y_train,
        n_classes
    )

    # Final Top-20 table and figure.
    final_top20_df = pd.DataFrame({
        "Rank": np.arange(
            1,
            len(final_fs["names"]) + 1
        ),
        "Feature": final_fs["names"],
        "RF_Importance": final_fs["importance"]
    })

    save_table(
        final_top20_df,
        dirs["tables"]
        / f"{dataset_key}_table_06_final_top20_features"
    )

    plot_top20_features(
        final_top20_df,
        f"{dataset_title}: Final RF-Selected Top 20 Features",
        dirs["figures"]
        / f"{dataset_key}_fig_03_top20_features"
    )

    plot_smote_distribution(
        y_train,
        y_train_bal,
        class_names,
        f"{dataset_title}: Training Distribution Before and After SMOTE",
        dirs["figures"]
        / f"{dataset_key}_fig_04_smote_distribution"
    )

    # ------------------------------------------------------------------
    # F. RETRAIN ONLY THE SELECTED TOP-3 + BUILD TEST META MATRIX
    # ------------------------------------------------------------------
    fresh_models = build_candidate_models(
        n_classes
    )

    final_selected_models = {}
    selected_test_probabilities = {}
    test_meta_parts = []
    final_rows = []

    for model_name in selected_models:
        model = fresh_models[
            model_name
        ]

        start = time.perf_counter()

        model.fit(
            X_train_bal,
            y_train_bal
        )

        fit_seconds = (
            time.perf_counter()
            - start
        )

        start = time.perf_counter()

        prob = model.predict_proba(
            X_test_scaled
        )

        inference_seconds = (
            time.perf_counter()
            - start
        )

        pred = np.argmax(
            prob,
            axis=1
        )

        metrics = calculate_metrics(
            y_test,
            pred,
            prob,
            n_classes
        )

        final_rows.append({
            "Model": model_name,
            **metrics,
            "Final_Training_Time_sec": fit_seconds,
            "Test_Inference_Time_sec": inference_seconds
        })

        final_selected_models[
            model_name
        ] = model

        selected_test_probabilities[
            model_name
        ] = prob

        test_meta_parts.append(
            probability_for_meta(
                prob,
                n_classes
            )
        )

    Z_test = np.hstack(
        test_meta_parts
    )

    # Meta-model was trained ONLY using training-set OOF predictions.
    # It is never fitted using final test observations.
    stack_prob = meta_model.predict_proba(
        Z_test
    )

    stack_pred = np.argmax(
        stack_prob,
        axis=1
    )

    stack_metrics = calculate_metrics(
        y_test,
        stack_pred,
        stack_prob,
        n_classes
    )

    final_rows.append({
        "Model": "Proposed Top-3 Stack",
        **stack_metrics,
        "Final_Training_Time_sec": np.nan,
        "Test_Inference_Time_sec": np.nan
    })

    selected_test_probabilities[
        "Proposed Top-3 Stack"
    ] = stack_prob

    final_test_df = pd.DataFrame(
        final_rows
    )

    save_table(
        final_test_df,
        dirs["tables"]
        / f"{dataset_key}_table_07_final_test_results"
    )

    # ------------------------------------------------------------------
    # G. FINAL PAPER FIGURES
    # ------------------------------------------------------------------
    plot_final_comparison(
        final_test_df,
        f"{dataset_title}: Final Untouched-Test Comparison",
        dirs["figures"]
        / f"{dataset_key}_fig_05_final_test_comparison"
    )

    plot_confusion(
        y_test,
        stack_pred,
        class_names,
        f"{dataset_title}: Proposed Stack Confusion Matrix",
        dirs["figures"]
        / f"{dataset_key}_fig_06_confusion_matrix",
        normalized=False
    )

    plot_confusion(
        y_test,
        stack_pred,
        class_names,
        f"{dataset_title}: Normalized Confusion Matrix",
        dirs["figures"]
        / f"{dataset_key}_fig_07_normalized_confusion_matrix",
        normalized=True
    )

    plot_roc(
        y_test,
        selected_test_probabilities,
        class_names,
        f"{dataset_title}: ROC Analysis",
        dirs["figures"]
        / f"{dataset_key}_fig_08_roc_curve"
    )

    plot_pr(
        y_test,
        selected_test_probabilities,
        class_names,
        f"{dataset_title}: Precision-Recall Analysis",
        dirs["figures"]
        / f"{dataset_key}_fig_09_pr_curve"
    )

    plot_meta_coefficients(
        meta_model,
        selected_models,
        n_classes,
        class_names,
        f"{dataset_title}: Logistic Regression Meta-Learner",
        dirs["figures"]
        / f"{dataset_key}_fig_10_meta_coefficients"
    )

    # ------------------------------------------------------------------
    # H. SHAP
    # ------------------------------------------------------------------
    shap_model_name = choose_shap_model(
        selected_models
    )

    shap_model = final_selected_models[
        shap_model_name
    ]

    generate_shap_analysis(
        shap_model,
        shap_model_name,
        X_test_scaled,
        final_fs["names"],
        class_names,
        dirs,
        dataset_key,
        dataset_title
    )

    # ------------------------------------------------------------------
    # I. CLASS-WISE RESULTS + PREDICTIONS
    # ------------------------------------------------------------------
    report = classification_report(
        y_test,
        stack_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_df = (
        pd.DataFrame(report)
        .T
        .reset_index()
        .rename(
            columns={
                "index": "Class_or_Average"
            }
        )
    )

    save_table(
        report_df,
        dirs["tables"]
        / f"{dataset_key}_table_09_classification_report"
    )

    prediction_df = pd.DataFrame({
        "True_Class": [
            class_names[i]
            for i in y_test
        ],
        "Predicted_Class": [
            class_names[i]
            for i in stack_pred
        ]
    })

    for c, cname in enumerate(
        class_names
    ):
        prediction_df[
            f"P_{cname}"
        ] = stack_prob[:, c]

    prediction_df.to_csv(
        dirs["reports"]
        / f"{dataset_key}_test_predictions.csv",
        index=False
    )

    # ------------------------------------------------------------------
    # J. TEXT/JSON SUMMARY FOR PAPER WRITING
    # ------------------------------------------------------------------
    experiment_summary = {
        "dataset": dataset_title,
        "records": int(len(X)),
        "training_records": int(len(X_train)),
        "untouched_test_records": int(len(X_test)),
        "features_before_selection": int(X.shape[1]),
        "final_top20_features": list(
            map(str, final_fs["names"])
        ),
        "selected_top3_models": selected_models,
        "selection_rule": (
            "Mean 5-fold Macro-F1; tie-breakers: MCC, "
            "Balanced Accuracy, PR-AUC"
        ),
        "meta_oof_cv_metrics": {
            k: float(v)
            for k, v in meta_cv_metrics.items()
        },
        "final_stack_test_metrics": {
            k: float(v)
            for k, v in stack_metrics.items()
        },
        "shap_explained_model": shap_model_name,
        "cleaning_info": cleaning_info
    }

    with open(
        dirs["reports"]
        / f"{dataset_key}_experiment_summary.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            experiment_summary,
            f,
            indent=2
        )

    summary_lines = [
        dataset_title,
        "=" * len(dataset_title),
        f"Total records: {len(X):,}",
        f"Training/Test: {len(X_train):,} / {len(X_test):,}",
        "",
        "Selected Top-3:",
        *[
            f"{i+1}. {name}"
            for i, name in enumerate(
                selected_models
            )
        ],
        "",
        f"OOF Meta-CV Accuracy: {meta_cv_metrics['Accuracy']:.6f}",
        f"OOF Meta-CV Macro-F1: {meta_cv_metrics['F1_Macro']:.6f}",
        "",
        f"Final Test Accuracy: {stack_metrics['Accuracy']:.6f}",
        f"Final Test Balanced Accuracy: {stack_metrics['Balanced_Accuracy']:.6f}",
        f"Final Test Macro-F1: {stack_metrics['F1_Macro']:.6f}",
        f"Final Test MCC: {stack_metrics['MCC']:.6f}",
        f"Final Test ROC-AUC: {stack_metrics['ROC_AUC']:.6f}",
        f"Final Test PR-AUC: {stack_metrics['PR_AUC']:.6f}",
        "",
        "Final RF Top-20 Features:",
        *[
            f"{i+1}. {feature}"
            for i, feature in enumerate(
                final_fs["names"]
            )
        ]
    ]

    Path(
        dirs["reports"]
        / f"{dataset_key}_paper_summary.txt"
    ).write_text(
        "\n".join(summary_lines),
        encoding="utf-8"
    )

    print("\nFinal Top-3:", selected_models)
    print(
        "Final Test Accuracy:",
        f"{stack_metrics['Accuracy']:.6f}"
    )
    print(
        "Final Test Macro-F1:",
        f"{stack_metrics['F1_Macro']:.6f}"
    )
    print(
        "Final Test MCC:",
        f"{stack_metrics['MCC']:.6f}"
    )

    return {
        "dataset_key": dataset_key,
        "dataset_title": dataset_title,
        "records": len(X),
        "train_records": len(X_train),
        "test_records": len(X_test),
        "selected_top3": selected_models,
        "ranking": ranking_df,
        "top20": list(final_fs["names"]),
        "meta_cv_metrics": meta_cv_metrics,
        "final_test_metrics": stack_metrics,
        "final_test_table": final_test_df,
        "output_dir": dirs["root"]
    }


# ======================================================================
# 14. LOAD ACTUAL DATASETS
# ======================================================================

primary_path = find_input_file(
    PRIMARY_FILENAME
)
secondary_path = find_input_file(
    SECONDARY_FILENAME
)

print("Primary:", primary_path)
print("Secondary:", secondary_path)

X_primary, y_primary, primary_classes, primary_cleaning = (
    load_primary_dataset(
        primary_path
    )
)

X_secondary, y_secondary, secondary_classes, secondary_cleaning = (
    load_secondary_dataset(
        secondary_path
    )
)


# ======================================================================
# 15. RUN BOTH EXPERIMENTS
# ======================================================================

primary_result = run_experiment(
    X_primary,
    y_primary,
    primary_classes,
    dataset_key="01_CICIDS2017_DNS_Binary",
    dataset_title="CICIDS2017 DNS-Associated Binary Detection",
    cleaning_info=primary_cleaning
)

secondary_result = run_experiment(
    X_secondary,
    y_secondary,
    secondary_classes,
    dataset_key="02_Simulated_DNS_Multiclass",
    dataset_title="Simulated DNS Multiclass Detection",
    cleaning_info=secondary_cleaning
)


# ======================================================================
# 16. COMBINED PAPER SUMMARY
# ======================================================================

combined_dir = (
    OUTPUT_ROOT
    / "00_Combined_Summary"
)
combined_dir.mkdir(
    parents=True,
    exist_ok=True
)

combined_rows = []

for result in [
    primary_result,
    secondary_result
]:
    m = result[
        "final_test_metrics"
    ]
    cv = result[
        "meta_cv_metrics"
    ]

    combined_rows.append({
        "Dataset": result[
            "dataset_title"
        ],
        "Records": result[
            "records"
        ],
        "Training": result[
            "train_records"
        ],
        "Test": result[
            "test_records"
        ],
        "Selected_Model_1": result[
            "selected_top3"
        ][0],
        "Selected_Model_2": result[
            "selected_top3"
        ][1],
        "Selected_Model_3": result[
            "selected_top3"
        ][2],
        "OOF_Meta_CV_Accuracy": cv[
            "Accuracy"
        ],
        "OOF_Meta_CV_F1_Macro": cv[
            "F1_Macro"
        ],
        "Test_Accuracy": m[
            "Accuracy"
        ],
        "Test_Balanced_Accuracy": m[
            "Balanced_Accuracy"
        ],
        "Test_Precision_Macro": m[
            "Precision_Macro"
        ],
        "Test_Recall_Macro": m[
            "Recall_Macro"
        ],
        "Test_F1_Macro": m[
            "F1_Macro"
        ],
        "Test_MCC": m[
            "MCC"
        ],
        "Test_ROC_AUC": m[
            "ROC_AUC"
        ],
        "Test_PR_AUC": m[
            "PR_AUC"
        ]
    })

combined_df = pd.DataFrame(
    combined_rows
)

save_table(
    combined_df,
    combined_dir
    / "table_01_combined_final_results"
)

# Side-by-side final Top-20 features.
feature_comparison = pd.DataFrame({
    "Rank": np.arange(
        1,
        TOP_K + 1
    ),
    "CICIDS2017": (
        primary_result[
            "top20"
        ]
        + [""] * (
            TOP_K
            - len(
                primary_result[
                    "top20"
                ]
            )
        )
    )[:TOP_K],
    "Simulated_DNS": (
        secondary_result[
            "top20"
        ]
        + [""] * (
            TOP_K
            - len(
                secondary_result[
                    "top20"
                ]
            )
        )
    )[:TOP_K]
})

save_table(
    feature_comparison,
    combined_dir
    / "table_02_top20_feature_comparison"
)

# Candidate ranking tables combined.
for result in [
    primary_result,
    secondary_result
]:
    result["ranking"].to_csv(
        combined_dir
        / (
            result["dataset_key"]
            + "_candidate_ranking.csv"
        ),
        index=False
    )

# Final concise text report.
report_lines = [
    "FINAL PAPER-READY EXPERIMENT SUMMARY",
    "=" * 75,
    ""
]

for result in [
    primary_result,
    secondary_result
]:
    m = result[
        "final_test_metrics"
    ]
    cv = result[
        "meta_cv_metrics"
    ]

    report_lines.extend([
        result[
            "dataset_title"
        ],
        "-" * len(
            result[
                "dataset_title"
            ]
        ),
        "Selected Top-3: "
        + ", ".join(
            result[
                "selected_top3"
            ]
        ),
        f"OOF Meta-CV Accuracy: {cv['Accuracy']:.6f}",
        f"OOF Meta-CV Macro-F1: {cv['F1_Macro']:.6f}",
        f"Final Test Accuracy: {m['Accuracy']:.6f}",
        f"Final Test Balanced Accuracy: {m['Balanced_Accuracy']:.6f}",
        f"Final Test Macro-F1: {m['F1_Macro']:.6f}",
        f"Final Test MCC: {m['MCC']:.6f}",
        f"Final Test ROC-AUC: {m['ROC_AUC']:.6f}",
        f"Final Test PR-AUC: {m['PR_AUC']:.6f}",
        ""
    ])

Path(
    combined_dir
    / "FINAL_RESULT_SUMMARY.txt"
).write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8"
)

# Zip the complete publication-output directory.
zip_base = "/kaggle/working/DNS_Paper_Results"
shutil.make_archive(
    zip_base,
    "zip",
    OUTPUT_ROOT
)

print("\n" + "=" * 90)
print("ALL EXPERIMENTS COMPLETED")
print("=" * 90)
print(combined_df.to_string(index=False))
print("\nResults folder:")
print(OUTPUT_ROOT)
print("\nZIP archive:")
print(zip_base + ".zip")

Primary: /kaggle/input/datasets/naimulpathan/dnsdataset/CICIDS2017 Dataset.csv
Secondary: /kaggle/input/datasets/naimulpathan/dnsdataset/CDAD_multiclass.csv
Automatically excluded non-behavioral / leakage-prone fields:
  - Experiment_ID
  - Session_ID
  - Scenario_ID
  - Record_Timestamp
  - Domain_Queried
  - Attack_Type

CICIDS2017 DNS-Associated Binary Detection
Records: 161,368
Raw predictive columns: 80
Classes: ['Benign', 'Attack']
Distribution: {np.int64(0): 161299, np.int64(1): 69}

Fold 1/5
  Before SMOTE: {np.int64(0): 103231, np.int64(1): 44}
  After SMOTE : {np.int64(0): 103231, np.int64(1): 25807}
  Logistic Regression  F1=0.92304  MCC=0.85628  Acc=0.99985
  SVM                  F1=0.92304  MCC=0.85628  Acc=0.99985
  Random Forest        F1=1.00000  MCC=1.00000  Acc=1.00000
  Extra Trees          F1=1.00000  MCC=1.00000  Acc=1.00000
  AdaBoost             F1=1.00000  MCC=1.00000  Acc=1.00000
  XGBoost              F1=0.92304  MCC=0.85628  Acc=0.99985
  LightGBM            